In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
import importlib
import rasterio

_training = importlib.import_module("30_Training")
MyGRU = _training.MyGRU


In [2]:
TEST_TRUTHS_PATH = "/home/main/repositories/RemoteSensing/MachineLearning/dataset/TestArea/cropped_annual_disturbances_1985_2023_slovakia.tif"
TEST_SPECTRAIL_INVENTORY = "/home/main/repositories/RemoteSensing/Qarterly_mosaics_30m_TestArea/inventory.csv"
TEST_FOREST_MASK_PATH = "/home/main/repositories/RemoteSensing/MachineLearning/dataset/TestArea/cropped_forest_mask_slovakia.tif"

MODEL_PATH = "/home/main/repositories/RemoteSensing/MachineLearning/Model_20261110225617/BestModel.pt"

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

loss_fn = nn.BCEWithLogitsLoss()

#load_model
checkpoint = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
state = checkpoint['model_state_dict']

model = MyGRU(loss_fn)
model.load_state_dict(state)
model.to(device)
model.eval()

MyGRU(
  (gru): GRU(4, 32, batch_first=True)
  (head): Linear(in_features=32, out_features=1, bias=True)
  (loss_fn): BCEWithLogitsLoss()
)

In [4]:
inventory = pd.read_csv(TEST_SPECTRAIL_INVENTORY, sep=";")
years = sorted(inventory["Year"].unique())
bands = sorted(inventory["Band"].unique())
#y_year = years[2:-2] #2017..2023

inventory = inventory.set_index(["Year","Band"]).sort_index()

In [5]:
with rasterio.open(TEST_TRUTHS_PATH, "r") as src:
    truhs = np.array(src.read())

with rasterio.open(inventory.iloc[0]["Path"], "r") as src:
    profile = src.profile

with rasterio.open(TEST_FOREST_MASK_PATH, "r") as src:
    f_mask = np.array(src.read(1))

In [10]:
def get_x_for_year(year, inve):
    ret=np.empty((truhs.shape[1],truhs.shape[2],4,4)) #[x,y,year,b]

    for i, y in enumerate(range(year-2, year+2)):
        for ii, b in enumerate(bands):
            with rasterio.open(inve.loc[(y,b)]["Path"]) as src:
                ret[:,:,i,ii] = src.read(1)/1000
    return torch.tensor( ret, dtype=torch.float32 )

def get_y_for_year(year):
    index_year = year - 1985
    return torch.tensor( truhs[index_year] )


In [16]:
total_correct=0
total = 0

THRESHOLD = 0.5

with torch.no_grad():
    for year in (range(2017, 2024)):
        x_nD=get_x_for_year(year, inventory)
        x = x_nD.reshape(-1, 4, 4)
        x = x.to(device)
        y=get_y_for_year(year)

        logits = model(x)
        probabilities = torch.sigmoid(logits)
        probabilities = probabilities.cpu().numpy()
        probabilities = probabilities.reshape(
            truhs.shape[1],
            truhs.shape[2]
        )

        predictions = np.zeros(shape=(y.shape))
        predictions[probabilities > THRESHOLD] = 1

        #mask
        probabilities[f_mask!=1] = profile['nodata']
        predictions[f_mask!=1] = profile['nodata']
        predictions[predictions == 0] = profile['nodata']

        #save
        prop_path = f"/home/main/repositories/RemoteSensing/MachineLearning/Model_20261110225617/Test_predictions/{year}_probability.gtiff"
        pred_path = f"/home/main/repositories/RemoteSensing/MachineLearning/Model_20261110225617/Test_predictions/{year}_prediction_T{THRESHOLD}.gtiff"

        with rasterio.open(prop_path, 'w', **profile) as dest:
            dest.write(probabilities,1)
        with rasterio.open(pred_path, 'w', **profile) as dest:
            dest.write(predictions,1)

